In [ ]:
from io import StringIO
import os 
from dotenv import load_dotenv

import climpred
import xarray as xr
import xesmf as xe
import numpy as np
import pandas as pda
import regionmask
import geopandas as gp
from climpred import HindcastEnsemble
from datetime import datetime

import xhistogram.xarray as xhist
from sklearn.metrics import roc_auc_score
import pandas as pd


import xskillscore as xs
from xbootstrap import block_bootstrap
from dask.distributed import Client

import altair as alt


import numpy as np
import pandas as pd

import pandas as pd
import numpy as np
import matplotlib

import os 
from dotenv import load_dotenv

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import six
from datetime import datetime
import textwrap as tw
from functools import reduce
import json

In [ ]:
load_dotenv()

data_path=os.getenv("data_path")

latex_path=os.getenv("latex_path")

# Data process functions

In [ ]:
def ken_mask_creator():
    """
    Utiliity for generating region/district masks using regionmask library

    Returns
    -------
    the_mask : TYPE
        DESCRIPTION.
    rl_dict : TYPE
        DESCRIPTION.

    """
    dis=gp.read_file(f'{data_path}Karamoja_boundary_dissolved.shp')
    mbt_path=os.getenv("mbt_path")
    reg=gp.read_file(f'{data_path}wajir_mbt_extent.shp')
    mds=pd.concat([dis,reg])
    mds1=mds.reset_index()
    mds1['region']=[0,1,2]
    mds1['region_name']=['Karamoja', 'Marsabit','Wajir']
    mds2=mds1[['geometry','region','region_name']]
    rl_dict=dict(zip(mds2.region, mds2.region_name))
    the_mask = regionmask.from_geopandas(mds2,numbers='region',overlap=True)
    return the_mask, rl_dict, mds2

def spi3_prod_name_creator(ds_ens,var_name):
    """
    Convenience function to generate a list of SPI product
    names, such as MAM, so that can be used to filter the 
    SPI product from dataframe

    added with method to convert the valid_time in CF format into datetime at
    line 3, which is the format given by climpred valid_time calculation 

    Parameters
    ----------
    ds_ens : xarray dataframe
        The data farme with SPI output organized for 
        the period 1981-2023.

    Returns
    -------
    spi_prod_list : String list
        List of names with iteration of SPI3 product names such as
        ['JFM','FMA','MAM',......]

    """
    db=pd.DataFrame()
    db['dt']=ds_ens[var_name].values
    db['dt1'] = db['dt'].apply(lambda x: datetime(x.year, x.month, x.day,
                                                                     x.hour, x.minute, x.second))
    #db['dt1']=db['dt'].to_datetimeindex()
    db['month']=db['dt1'].dt.strftime('%b').astype(str).str[0]
    db['year']=db['dt1'].dt.strftime('%Y')
    db['spi_prod'] = db.groupby('year')['month'].shift(2)+db.groupby('year')['month'].shift(1) + db.groupby('year')['month'].shift(0)
    spi_prod_list=db['spi_prod'].tolist()
    return spi_prod_list


def spi4_prod_name_creator(ds_ens,var_name):
    """
    Convenience function to generate a list of SPI product
    names, such as MAM, so that can be used to filter the 
    SPI product from dataframe

    added with method to convert the valid_time in CF format into datetime at
    line 3, which is the format given by climpred valid_time calculation 

    Parameters
    ----------
    ds_ens : xarray dataframe
        The data farme with SPI output organized for 
        the period 1981-2023.

    Returns
    -------
    spi_prod_list : String list
        List of names with iteration of SPI3 product names such as
        ['JFM','FMA','MAM',......]

    """
    db=pd.DataFrame()
    db['dt']=ds_ens[var_name].values
    db['dt1'] = db['dt'].apply(lambda x: datetime(x.year, x.month, x.day,
                                                                     x.hour, x.minute, x.second))
    #db['dt1']=db['dt'].to_datetimeindex()
    db['month']=db['dt1'].dt.strftime('%b').astype(str).str[0]
    db['year']=db['dt1'].dt.strftime('%Y')
    db['spi_prod'] = db.groupby('year')['month'].shift(3)+db.groupby('year')['month'].shift(2)+db.groupby('year')['month'].shift(1) + db.groupby('year')['month'].shift(0)
    spi_prod_list=db['spi_prod'].tolist()
    return spi_prod_list


def make_obs_fct_dataset(region_id,season_str,lead_int):
    """
    Prepares observed and forecasted dataset subsets for a specific region, season, and lead time.

    This function loads observed and forecasted datasets based on the season string length (indicating SPI3 or SPI4),
    applies regional masking, selects the data for the given region by its ID, and subsets the data for the specified
    season and lead time. It then aligns the observed dataset time coordinates with the forecasted dataset valid time
    coordinates and returns both datasets.

    Parameters:
    - region_id (int): The identifier for the region of interest.
    - season_str (str): A string representing the season. The length of this string determines whether SPI3 or SPI4
                        datasets are used ('mam', 'jjas', etc. for SPI3, and longer strings for SPI4).
    - lead_int (int): The lead time index for which the forecast dataset is to be subset.

    Returns:
    - obs_data (xarray.DataArray): The subsetted observed data array for the specified region, season, and aligned time coordinates.
    - ens_data (xarray.DataArray): The subsetted forecast data array for the specified region, season, lead time, and aligned time coordinates.

    Notes:
    - The function assumes the existence of a `data_path` variable that specifies the base path to the dataset files.
    - It requires the `xarray` library for data manipulation and assumes specific naming conventions for the dataset files.
    - Regional masking and season-specific processing rely on externally defined functions and naming conventions.
    - The final alignment of observed dataset time coordinates with forecasted dataset valid time coordinates ensures
      comparability between observed and forecasted values for verification purposes.

    Example Usage:
    >>> obs_data, ens_data = make_obs_fct_dataset(1, 'mam', 0)
    >>> print(obs_data)
    >>> print(ens_data)

    This would load the observed and forecasted SPI3 datasets for region 1 during the 'mam' season and subset them
    for lead time index 0, aligning the observed data time coordinates with the forecasted data valid time coordinates.
    """
    if len(season_str) == 3:
        kn_fct=xr.open_dataset(f'{data_path}kn_fct_spi3.nc')
        kn_obs=xr.open_dataset(f'{data_path}kn_obs_spi3.nc')
    else:
        kn_fct=xr.open_dataset(f'{data_path}kn_fct_spi4.nc')
        kn_obs=xr.open_dataset(f'{data_path}kn_obs_spi4.nc')
    the_mask, rl_dict,mds1=ken_mask_creator()
    bounds = mds1.bounds
    #bounds.iloc[0].minx
    llon=bounds.iloc[region_id].minx
    llat=bounds.iloc[region_id].miny
    ulon=bounds.iloc[region_id].maxx
    ulat=bounds.iloc[region_id].maxy
    a_fc=kn_fct.sel(lon=slice(llon, ulon), lat=slice(llat,ulat))
    a_obs=kn_obs.sel(lon=slice(llon, ulon), lat=slice(llat,ulat))
    hindcast = HindcastEnsemble(a_fc)
    hindcast = hindcast.add_observations(a_obs)
    #hindcast
    #spi_cdb1spi3_prod_name_creator(ds_ens)
    a_fc1=hindcast.get_initialized()
    a_fc2=a_fc1.isel(lead=lead_int)
    if len(season_str) == 3:
        spi_prod_list=spi3_prod_name_creator(a_fc2,'valid_time')
        obs_spi_prod_list=spi3_prod_name_creator(a_obs,'time')
    else:
        spi_prod_list=spi4_prod_name_creator(a_fc2,'valid_time')
        obs_spi_prod_list=spi4_prod_name_creator(a_obs,'time')
    a_fc2 = a_fc2.assign_coords(spi_prod=('init',spi_prod_list))
    a_fc3=a_fc2.where(a_fc2.spi_prod==season_str, drop=True)
    #obsertations
    a_obs1 = a_obs.assign_coords(spi_prod=('time',obs_spi_prod_list))
    a_obs2=a_obs1.where(a_obs1.spi_prod==season_str, drop=True)
    #valid_time_series = a_fc3.valid_time.to_series().reset_index(drop=True).drop_duplicates()
    valid_time_flattened = a_fc2.valid_time.to_dataframe().reset_index().drop_duplicates(subset='valid_time')['valid_time']
    valid_time_flattened.columns=['valid_time','cc']
    #valid_time_flattened['valid_time'] = pd.to_datetime(valid_time_flattened['valid_time'])
    # Apply lambda function to create 'dt1' column
    #valid_time_flattened['dt1'] = valid_time_flattened['valid_time'].apply(
    #    lambda x: datetime(x.year, x.month, x.day, x.hour, x.minute, x.second)
    #)
    #
    valid_time_flattened['dt1'] =valid_time_flattened['valid_time'].apply(lambda x: datetime(x.year, x.month, x.day,x.hour, x.minute, x.second))
    # Ensure the valid_time is in 'YYYY-MM-DD' string format
    #valid_time_flattened['dt2'] = valid_time_flattened['dt1'].dt.strftime('%Y-%m-%d')
    valid_time_flattened['dt1'] = valid_time_flattened['dt1'].dt.strftime('%Y-%m-%dT%H:%M:%S.%f')
    valid_time_flattened['dt1'] = pd.to_datetime(valid_time_flattened['dt1'])
    # Convert to xarray DataArray with time as the dimension name
    #valid_time_da = xr.DataArray(valid_time_flattened['dt1'], dims=['time'])
    valid_time_da = xr.DataArray(valid_time_flattened['dt1'], dims=['time'],coords=valid_time_flattened['dt1'])
    a_obs3 = a_obs2.reindex(time=valid_time_da)
    #a_obs4 = a_obs3.reindex(time=a_obs2.time)
    #a_obs4 = a_obs3.sel(time=a_obs2.time, drop=True)
    a_obs3 = a_obs3.dropna(dim='time')
    if len(season_str) == 3:
        obs_data=a_obs3['spi3']
        ens_data=a_fc3['spi3']
    else:
        obs_data=a_obs3['spi4']
        ens_data=a_fc3['spi4']
    return obs_data, ens_data


def get_threshold(region_id, season):
    """
    Retrieves the drought threshold value for a specified region, season, and drought level.

    The function reads predefined threshold values from a CSV-format string. It looks up the threshold for the given
    region ID, season, and drought level ('mod' for moderate, 'sev' for severe, or 'ext' for extreme). These thresholds
    are specific to certain regions and seasons and indicate the level at which a drought event of a particular severity
    is considered to occur.

    Parameters:
    - region_id (int): The integer identifier for the region of interest.
    - season (str): The season for which the threshold is required. Expected values are season codes such as 'mam' (March-April-May),
                    'jjas' (June-July-August-September), 'ond' (October-November-December), etc.
    - level (str): The drought severity level for which the threshold is requested. Valid options are 'mod' for moderate,
                   'sev' for severe, and 'ext' for extreme drought conditions.

    Returns:
    - float: The threshold value for the specified region, season, and drought level. Returns None if no threshold is found for the given inputs.

    Note:
    - This function uses a hardcoded CSV string as its data source. In a production environment, it's recommended to
      store and retrieve such data from a more robust data management system.
    - The function requires the pandas library for data manipulation and the StringIO module from io for string-based data input.

    Example usage:
    >>> threshold = get_threshold(1, 'mam', 'mod')
    >>> print(threshold)
    -0.14
    """
    data = """region_id,region,season,mod,sev,ext
    0,kmj,mam,-0.03,-0.56,-0.99
    0,kmj,jjas,-0.01,-0.41,-0.99
    1,mbt,mam,-0.14,-0.38,-0.8
    1,mbt,ond,-0.15,-0.53,-0.71
    2,wjr,mam,-0.19,-0.45,-0.75
    2,wjr,ond,-0.29,-0.76,-0.9
    """
    # Use StringIO to convert the string data to a file-like object
    data_io = StringIO(data)
    # Read the data into a pandas DataFrame
    df = pd.read_csv(data_io)
    thresholds_dict = { (row['region_id'], row['season']): {'mod': row['mod'], 'sev': row['sev'], 'ext': row['ext']}
                   for _, row in df.iterrows() }
    # Retrieve the dictionary for the given region_id and season
    season_thresholds = thresholds_dict.get((region_id, season), {})
    # Return the threshold for the given level (mod, sev, ext), or None if not found
    return season_thresholds

def mean_obs_spi(obs_data, spi_string_name):
    obs_data_mean = obs_data.mean(dim=["lat", "lon"])
    obs_data_df = obs_data_mean.to_dataframe().reset_index()
    obs_data_df1 = obs_data_df[["time", spi_string_name]]
    wdf=obs_data_df1
    wdf["year0"] = wdf["time"].apply(
        lambda x: datetime(x.year, x.month, x.day, x.hour, x.minute, x.second)
    )
    wdf["year"] = wdf["year0"].dt.strftime("%Y")
    wdf1 = wdf[[spi_string_name, "year"]]
    return wdf1

def emprical_probablity(ens_data, threshold_dict):
    mod_thr = threshold_dict["mod"]
    fct_mod = (ens_data <= mod_thr).mean(dim="member")
    ####
    sev_thr = threshold_dict["sev"]
    fct_sev = (ens_data <= sev_thr).mean(dim="member")
    ####
    ext_thr = threshold_dict["ext"]
    fct_ext = (ens_data <= ext_thr).mean(dim="member")
    return fct_mod, fct_sev, fct_ext

def mean_emp_prob(fct_mod, fct_sev, fct_ext, spi_string_name):
    fct_mod_mean = fct_mod.mean(dim=["lat", "lon"])
    fct_mod_df = fct_mod_mean.to_dataframe().reset_index()
    fct_mod_df1 = fct_mod_df[["valid_time", spi_string_name]]
    fct_mod_df1 = fct_mod_df1.assign(cat="mod")
    fct_sev_mean = fct_sev.mean(dim=["lat", "lon"])
    fct_sev_df = fct_sev_mean.to_dataframe().reset_index()
    fct_sev_df1 = fct_sev_df[["valid_time", spi_string_name]]
    fct_sev_df1 = fct_sev_df1.assign(cat="sev")
    fct_ext_mean = fct_ext.mean(dim=["lat", "lon"])
    fct_ext_df = fct_ext_mean.to_dataframe().reset_index()
    fct_ext_df1 = fct_ext_df[["valid_time", spi_string_name]]
    fct_ext_df1 = fct_ext_df1.assign(cat="ext")
    wdf = pd.concat([fct_mod_df1, fct_sev_df1, fct_ext_df1])
    wdf["year0"] = wdf["valid_time"].apply(
        lambda x: datetime(x.year, x.month, x.day, x.hour, x.minute, x.second)
    )
    wdf["year"] = wdf["year0"].dt.strftime("%Y")
    wdf1 = wdf[[spi_string_name, "cat", "year"]]
    wdf1.columns=['ep','cat','year']
    #wdf1['ep']=wdf1['ep']*100
    wdf1.loc[:, 'ep'] = wdf1['ep'] * 100
    return wdf1


# Calculate AUROC using bootstrap
def calculate_auroc(hits, misses, false_alarms, correct_negatives):
    """
    Calculates the Area Under the Receiver Operating Characteristic (AUROC) curve for a set of forecasts relative to observations.

    This function computes the AUROC score as a measure of the forecast's ability to discriminate between two classes:
    events that occurred (drought) and events that did not occur (no drought). The AUROC score ranges from 0 to 1,
    where a score of 0.5 suggests no discriminative ability (equivalent to random chance), and a score of 1 indicates perfect discrimination.

    Parameters:
    - hits (int): The number of correctly forecasted events (true positives).
    - misses (int): The number of events that were observed but not forecasted (false negatives).
    - false_alarms (int): The number of non-events that were incorrectly forecasted as events (false positives).
    - correct_negatives (int): The number of non-events that were correctly forecasted (true negatives).

    Returns:
    - auroc (float): The calculated AUROC score for the given contingency table values.

    Note:
    - This function is designed to work with binary classification problems, such as predicting the occurrence or non-occurrence of drought events.
    - It requires the `roc_auc_score` function from the `sklearn.metrics` module and `numpy` for handling arrays.

    Example usage:
    >>> auroc_score = calculate_auroc(50, 30, 20, 100)
    >>> print(f"AUROC Score: {auroc_score}")
    """
    total_positives = hits + misses
    total_negatives = correct_negatives + false_alarms
    y_true = np.concatenate((np.ones(total_positives), np.zeros(total_negatives)))
    y_scores = np.concatenate((np.ones(hits), np.zeros(misses + false_alarms + correct_negatives)))
    auroc = roc_auc_score(y_true, y_scores)
    return auroc



def xhist_metrices(pdb,trigger_value,threshold_dict,cat_str):
    ds = xr.Dataset.from_dataframe(pdb)
    obs_ext = ds[f'spi3_{cat_str}']
    fct_ext = ds[f'ep_{cat_str}']
    obs_event = obs_ext <= threshold_dict[cat_str]
    fct_event = fct_ext >= trigger_value
    obs_event_int=obs_event.astype(int)
    fct_event_int=fct_event.astype(int)
    contingency_table = xhist.histogram(
            obs_event_int,
            fct_event_int,
            bins=[2, 2],
            density=False
        )
    contingency_table = contingency_table.data
    correct_negatives = contingency_table[0, 0]
    false_alarms = contingency_table[0, 1]
    misses = contingency_table[1, 0]
    hits = contingency_table[1, 1]
    total = hits + false_alarms + misses + correct_negatives
    hit_rates = hits / (hits + misses) if (hits + misses) > 0 else np.nan
    false_alarm_ratios = false_alarms / (false_alarms + hits) if (false_alarms + hits) > 0 else np.nan
        #false_alarm_ratios[i] = false_alarms / (false_alarms + correct_negatives) if (false_alarms + correct_negatives) > 0 else np.nan
    bias_scores = (hits + false_alarms) / (hits + misses) if (hits + misses) > 0 else np.nan
    n_hit_rates = np.mean(hits.astype(int))  # Calculate hit rate as mean of hits
    n_false_alarm_ratios = np.mean(false_alarm_ratios.astype(int))
    hanssen_kuipers_scores = n_hit_rates - n_false_alarm_ratios
    heidke_skill_scores = (hits * correct_negatives - misses * false_alarms) / total
    
    fct_ext_pb=fct_ext/100
    tv_pb=trigger_value/100
    o1 = block_bootstrap(obs_event_int,blocks={"index": 1},n_iteration=1000,circular=True,)
    f1 = block_bootstrap(fct_ext_pb,blocks={"index": 1},n_iteration=1000,circular=True,)
    fpr, tpr, auroc_bootstrap_scores = xs.roc(o1, f1, bin_edges=[0,tv_pb,1],dim=['index'],return_results='all_as_metric_dim')
    auroc_scores = np.mean(auroc_bootstrap_scores)
    auroc_lb, auroc_ub = np.percentile(auroc_bootstrap_scores, [2.5, 97.5])
    df = pd.DataFrame({
        '#dry-seas':len(obs_ext.index.values),
        'hits': [hits],
        'misses':[misses],
        'FA': [false_alarms],
        'CN': [correct_negatives],
        'hit_rates': [hit_rates],
        'false_alarm_ratios': [false_alarm_ratios],
        'bias_scores': [bias_scores],
        'hanssen_kuipers_scores': [hanssen_kuipers_scores],
        'heidke_skill_scores': [heidke_skill_scores],
        'auroc_scores': auroc_scores.values,
        'auroc_lb': auroc_lb,
        'auroc_ub': auroc_ub
    })
    df.insert(0, 'threshold', threshold_dict[cat_str])
    df.insert(0, 'trigger_values', trigger_value)
    return df


def get_subset(dfa,cat_str):
    # Filter out rows with null values in 'hit_rate' and 'false_alarm_ratio'
    #df = df.dropna(subset=['hit_rate', 'false_alarm_ratio'])
    df=dfa[dfa['cat']==cat_str]
    # Sort the DataFrame by 'peirce_score' in descending order
    df = df.sort_values(by='hanssen_kuipers_scores', ascending=False)
    
    # Get the row with the maximum 'peirce_score'
    max_peirce_row = df.iloc[0]
    
    # Sort the DataFrame by 'bias_score' in descending order, and filter for 'bias_score' < 1.0
    df = df.loc[df['bias_scores'] < 1.0].sort_values(by='bias_scores', ascending=False)
    
    # Get the row with the maximum 'bias_score' < 1.0
    max_bias_row = df.iloc[0]
    
    # Sort the DataFrame by 'heidke_score' in descending order
    df = df.sort_values(by='heidke_skill_scores', ascending=False)
    
    # Get the row with the maximum 'heidke_score'
    max_heidke_row = df.iloc[0]
    
    # Combine the three rows into a subset
    subset = pd.concat([pd.DataFrame([max_peirce_row]), pd.DataFrame([max_bias_row]), pd.DataFrame([max_heidke_row])], ignore_index=True)
    
    return subset



def trigger_decision_dict(df0):
    df=df0[df0['auroc_scores']>=0.5]
    df_mod=get_subset(df,'mod')
    mod_max_cn = df_mod['CN'].max()
    mod_df_max_cn = df_mod[df_mod['CN'] == mod_max_cn]
    mod_max_hits = mod_df_max_cn['hits'].max()
    mod_df_max_hits = mod_df_max_cn[mod_df_max_cn['hits'] == mod_max_hits]
    
    df_sev=get_subset(df,'sev')
    sev_max_cn = df_sev['CN'].max()
    sev_df_max_cn = df_sev[df_sev['CN'] == sev_max_cn]
    sev_max_hits = sev_df_max_cn['hits'].max()
    sev_df_max_hits = sev_df_max_cn[sev_df_max_cn['hits'] == sev_max_hits]
    
    df_ext=get_subset(df,'ext')
    ext_max_cn = df_ext['CN'].max()
    ext_df_max_cn = df_ext[df_ext['CN'] == ext_max_cn]
    ext_max_hits = ext_df_max_cn['hits'].max()
    ext_df_max_hits = ext_df_max_cn[ext_df_max_cn['hits'] == ext_max_hits]
    tri_dict={'mod':mod_df_max_hits['trigger_values'].values[0],
              'sev':sev_df_max_hits['trigger_values'].values[0],
              'ext':ext_df_max_hits['trigger_values'].values[0]}
    df0=pd.concat([mod_df_max_hits,sev_df_max_hits,ext_df_max_hits])
    return tri_dict,df0

   
def get_mean_ens_triggers(region_id, season_str, lead_int):
    if len(season_str) == 3:
        spi_string_name = "spi3"
    else:
        spi_string_name = "spi4"
    sc_season_str = season_str.lower()
    obs_data, ens_data = make_obs_fct_dataset(region_id, season_str, lead_int)
    obs_df = mean_obs_spi(obs_data, spi_string_name)
    threshold_dict = get_threshold(region_id, sc_season_str)
    fct_mod, fct_sev, fct_ext = emprical_probablity(ens_data, threshold_dict)
    fct_df = mean_emp_prob(fct_mod, fct_sev, fct_ext, spi_string_name)
    db = pd.merge(fct_df, obs_df, on="year")
    pdb = db.pivot(index="year", columns="cat", values=["spi3", "ep"])
    pdb.columns = ["{}_{}".format(val[0], val[1]) for val in pdb.columns]
    pdb1 = pdb[pdb["spi3_ext"] <= 0]
    pdb2=pdb.reset_index()
    cnt_df = []
    for idx, row in pdb2.iterrows():
        mod_trigger_value = row["ep_mod"]
        mod_df = xhist_metrices(pdb2, mod_trigger_value, threshold_dict, "mod")
        mod_df.insert(0, "region", region_id)
        mod_df.insert(1, "season", season_str)
        mod_df.insert(2, "cat", 'mod')
        mod_df.insert(3, "year", row["year"])
        cnt_df.append(mod_df)

        sev_trigger_value = row["ep_sev"]
        sev_df = xhist_metrices(pdb2, sev_trigger_value, threshold_dict, "sev")
        sev_df.insert(0, "region", region_id)
        sev_df.insert(1, "season", season_str)
        sev_df.insert(2, "cat", 'sev')
        sev_df.insert(3, "year", row["year"])
        cnt_df.append(sev_df)

        ext_trigger_value = row["ep_ext"]
        ext_df = xhist_metrices(pdb2, ext_trigger_value, threshold_dict, "ext")
        ext_df.insert(0, "region", region_id)
        ext_df.insert(1, "season", season_str)
        ext_df.insert(2, "cat", 'ext')
        ext_df.insert(3, "year", row["year"])
        cnt_df.append(ext_df)

    metrix_df = pd.concat(cnt_df)
    decision_dict, decision_df = trigger_decision_dict(metrix_df)
    decision_df['lead_time']=lead_int
    pdb_melt=pdb.rename(columns={"ep_ext": "ext", "ep_sev": "sev","ep_mod":"mod"})
    plot_df = pd.melt(pdb_melt.reset_index(), id_vars=['year'], value_vars=['mod', 'sev', 'ext'],
                    var_name='cat', value_name='ep_pb')
    return obs_df, fct_df, metrix_df, decision_dict, decision_df, plot_df


# Visualization functions 
## Altair 

In [ ]:
def obs_chart_with_triggers(plot_type, df, year_column, spi_column, threshold_dict,row_annotations):
    """
    Create an Altair chart with a bar chart overlaid by trigger lines.

    Parameters:
    df : pandas.DataFrame
        The DataFrame containing the data.
    year_column : str
        The name of the DataFrame column containing the year.
    spi_column : str
        The name of the DataFrame column containing SPI values.
    threshold_dict : dict
        A dictionary with keys as threshold names and values as threshold values.
    """

    # Bar chart
    if plot_type == 'obs':
        bar_chart = alt.Chart(df).mark_bar().encode(
            x=alt.X(f'{year_column}:N', axis=alt.Axis(labelAngle=90)),
            y=alt.Y(f'{spi_column}:Q', title=spi_column, scale=alt.Scale(domain=[-4, 4])),
            color=alt.condition(
                alt.datum[spi_column] > 0,
                alt.value('blue'),  # Color for positive values
                alt.value('red')  # Color for negative values
            )
        ).properties(
            width=400,
            height=200
        )
    else:
        color_scale = alt.Scale(
            # domain=["ext", "sev", "mod"], range=["#880203", "#ffa400", "#fffe00"]
            domain=["mod", "sev", "ext"],
            range=["#f4eb13", "#f89821", "#ed2227"],
        )

        bar_chart = alt.Chart(df).mark_bar().encode(
            x=alt.X(f'{year_column}:N', axis=alt.Axis(labelAngle=90)),
            y=alt.Y(f'{spi_column}:Q', title='Probability (%)', stack=None),
            color=alt.Color('cat:N', scale=color_scale, sort=["sev", "mod", "ext"]),
        ).properties(
            width=400,
            height=200
        )+row_annotations

    # Adding trigger lines
    rules = []
    for key, value in threshold_dict.items():
        rule = alt.Chart(pd.DataFrame({'y': [value]})).mark_rule(
            strokeWidth=2,
            stroke={'ext': '#ed2227', 'sev': '#f89821', 'mod': '#f4eb13'}[key]  # Conditional color assignment
        ).encode(
            y='y:Q'
        )
        rules.append(rule)

    # Combine the bar chart with trigger lines
    final_chart = alt.layer(bar_chart, *rules)

    return final_chart


def make_barchart_annotations():
    row_annotations = [
    alt.Chart(pd.DataFrame({'text': ['lt=1']})).mark_text(
        align='left',
        baseline='middle',
        fontSize=14,
        fontWeight='bold',
        dx=-190,
        dy=-90
    ).encode(
        text='text:N'
    ).properties(width=400, height=200),
    alt.Chart(pd.DataFrame({'text': ['lt=2, Sep']})).mark_text(
        align='left',
        baseline='middle',
        fontSize=14,
        fontWeight='bold',
        dx=-190,
        dy=-90
    ).encode(
        text='text:N'
    ).properties(width=400, height=200),
    alt.Chart(pd.DataFrame({'text': ['lt=3, Aug']})).mark_text(
        align='left',
        baseline='middle',
        fontSize=14,
        fontWeight='bold',
        dx=-190,
        dy=-90
    ).encode(
        text='text:N'
    ).properties(width=400, height=200),
    alt.Chart(pd.DataFrame({'text': ['lt=4, Jul']})).mark_text(
        align='left',
        baseline='middle',
        fontSize=14,
        fontWeight='bold',
        dx=-190,
        dy=-90
    ).encode(
        text='text:N'
    ).properties(width=400, height=200),
    alt.Chart(pd.DataFrame({'text': ['lt=5']})).mark_text(
        align='left',
        baseline='middle',
        fontSize=14,
        fontWeight='bold',
        dx=-190,
        dy=-90
    ).encode(
        text='text:N'
    ).properties(width=400, height=200)]
    return row_annotations

def table(df):
    return (
        alt.Chart(df.reset_index())
        .mark_text()
        .transform_fold(df.columns.tolist())
        .encode(
            alt.X(
                "key",
                type="nominal",
                axis=alt.Axis(
                    # flip x labels upside down
                    orient="top",
                    # put x labels into horizontal direction
                    labelAngle=0,
                    title=None,
                    ticks=False
                ),
                scale=alt.Scale(padding=10),
                sort=None,
            ),
            alt.Y("index", type="ordinal", axis=None),
            alt.Text("value", type="nominal"),
        )
    )

## matplotlib functions 

In [ ]:
def create_month_column(df):
    new_column = []
    
    for _, row in df.iterrows():
        lt = row['lt']
        cat = row['cat']
        season = row['season']
        
        if season == 'MAM':
            if lt == 1:
                if cat == 'mod':
                    new_column.append('mar_x')
                elif cat == 'sev':
                    new_column.append('mar_y')
                elif cat == 'ext':
                    new_column.append('mar_z')
            elif lt == 2:
                if cat == 'mod':
                    new_column.append('feb_x')
                elif cat == 'sev':
                    new_column.append('feb_y')
                elif cat == 'ext':
                    new_column.append('feb_z')
            elif lt == 3:
                if cat == 'mod':
                    new_column.append('jan_x')
                elif cat == 'sev':
                    new_column.append('jan_y')
                elif cat == 'ext':
                    new_column.append('jan_z')
            elif lt == 4:
                if cat == 'mod':
                    new_column.append('dec_x')
                elif cat == 'sev':
                    new_column.append('dec_y')
                elif cat == 'ext':
                    new_column.append('dec_z')
            elif lt == 5:
                if cat == 'mod':
                    new_column.append('nov_x')
                elif cat == 'sev':
                    new_column.append('nov_y')
                elif cat == 'ext':
                    new_column.append('nov_z')
        elif season == 'OND':
            if lt == 1:
                if cat == 'mod':
                    new_column.append('oct_x')
                elif cat == 'sev':
                    new_column.append('oct_y')
                elif cat == 'ext':
                    new_column.append('oct_z')
            elif lt == 2:
                if cat == 'mod':
                    new_column.append('sep_x')
                elif cat == 'sev':
                    new_column.append('sep_y')
                elif cat == 'ext':
                    new_column.append('sep_z')
            elif lt == 3:
                if cat == 'mod':
                    new_column.append('aug_x')
                elif cat == 'sev':
                    new_column.append('aug_y')
                elif cat == 'ext':
                    new_column.append('aug_z')
            elif lt == 4:
                if cat == 'mod':
                    new_column.append('jul_x')
                elif cat == 'sev':
                    new_column.append('jul_y')
                elif cat == 'ext':
                    new_column.append('jul_z')
            elif lt == 5:
                if cat == 'mod':
                    new_column.append('jun_x')
                elif cat == 'sev':
                    new_column.append('jun_y')
                elif cat == 'ext':
                    new_column.append('jun_z')
        elif season == 'JJAS':
            if lt == 2:
                if cat == 'mod':
                    new_column.append('jun_x')
                elif cat == 'sev':
                    new_column.append('jun_y')
                elif cat == 'ext':
                    new_column.append('jun_z')
            elif lt == 3:
                if cat == 'mod':
                    new_column.append('may_x')
                elif cat == 'sev':
                    new_column.append('may_y')
                elif cat == 'ext':
                    new_column.append('may_z')
            elif lt == 4:
                if cat == 'mod':
                    new_column.append('apr_x')
                elif cat == 'sev':
                    new_column.append('apr_y')
                elif cat == 'ext':
                    new_column.append('apr_z')
            elif lt == 5:
                if cat == 'mod':
                    new_column.append('mar_x')
                elif cat == 'sev':
                    new_column.append('mar_y')
                elif cat == 'ext':
                    new_column.append('mar_z')
        else:
            new_column.append('')
    
    df['new_column'] = new_column
    return df



def replace_with_list(x):
    """
    Replaces NaN float values with a predefined list of replacement values.

    Parameters:
    - x (float): The input value to be checked and potentially replaced.

    Returns:
    - A list of replacement values if `x` is a float and is NaN. Otherwise, returns `x` unchanged.

    Note:
    - This function is designed to handle cases where cell values in a dataset need to be replaced with a list of values
      for indicating missing or special cases.
    """
    replacement_values = [-999.0, -999.0]
    # If x is a float and it is nan (meaning the cell was originally empty), return the replacement list
    if isinstance(x, float) and np.isnan(x):
        return replacement_values
    # Otherwise, return x as it is
    return x


def round_list(lst, decimal_places):
    """
    Rounds each element in a list to a specified number of decimal places.

    Parameters:
    - lst (list of float): The list of numbers to be rounded.
    - decimal_places (int): The number of decimal places to round each number to.

    Returns:
    - A list containing the rounded values of the input list.

    Note:
    - This function is useful for rounding numerical values in a list to ensure consistency or to improve readability.
    """
    return [round(x, decimal_places) for x in lst]




# %% table plot matplotlib

### Define the picture size and remove the ticks


### functions for whole column, row editing
def legend_maker(text1, color_list, legend_title):
    square6 = plt.Rectangle((0.4, 0.1), 0.15, 0.25, color=color_list[0], clip_on=False)
    text1.add_artist(square6)
    square5 = plt.Rectangle((0.55, 0.1), 0.15, 0.25, color=color_list[1], clip_on=False)
    text1.add_artist(square5)
    square5 = plt.Rectangle((0.7, 0.1), 0.15, 0.25, color=color_list[2], clip_on=False)
    text1.add_artist(square5)
    square5 = plt.Rectangle((0.85, 0.1), 0.15, 0.25, color=color_list[3], clip_on=False)
    text1.add_artist(square5)
    square5 = plt.Rectangle((1.0, 0.1), 0.15, 0.25, color=color_list[4], clip_on=False)
    text1.add_artist(square5)
    plt.text(
        0.6,
        0.4,
        legend_title,
        horizontalalignment="left",
        fontsize=6,
        fontweight="bold",
        color="k",
        verticalalignment="center",
        transform=text1.transAxes,
    )
    plt.text(
        0.42,
        0.05,
        "<20",
        horizontalalignment="left",
        fontsize=6,
        fontweight="bold",
        color="k",
        verticalalignment="center",
        transform=text1.transAxes,
    )
    plt.text(
        0.57,
        0.05,
        "20-40",
        horizontalalignment="left",
        fontsize=6,
        fontweight="bold",
        color="k",
        verticalalignment="center",
        transform=text1.transAxes,
    )
    plt.text(
        0.72,
        0.05,
        "40-60",
        horizontalalignment="left",
        fontsize=6,
        fontweight="bold",
        color="k",
        verticalalignment="center",
        transform=text1.transAxes,
    )
    plt.text(
        0.87,
        0.05,
        "60-80",
        horizontalalignment="left",
        fontsize=6,
        fontweight="bold",
        color="k",
        verticalalignment="center",
        transform=text1.transAxes,
    )
    plt.text(
        1.05,
        0.05,
        "80<",
        horizontalalignment="left",
        fontsize=6,
        fontweight="bold",
        color="k",
        verticalalignment="center",
        transform=text1.transAxes,
    )


def set_align_for_column(table, col, align="left"):
    cells = [key for key in table._cells if key[1] == col]
    for cell in cells:
        table._cells[cell]._loc = align


def set_width_for_column(table, col, width):
    cells = [key for key in table._cells if key[1] == col]
    for cell in cells:
        table._cells[cell]._width = width


def set_height_for_row(table, row, height):
    cells = [key for key in table._cells if key[0] == row]
    for cell in cells:
        table._cells[cell]._height = height


def colorcell(tablerows, tablecols, cellDict, color_list):
    allcells = [(x, y) for x in tablerows[1:] for y in tablecols[2:]]
    for alcls in allcells:
        cell_value0 = json.loads(cellDict[alcls]._text.get_text())[0]
        if cell_value0 == -999.0:
            cellDict[alcls].set_facecolor("#FFFFFF")
        else:
            if float(cell_value0) <= 0.2:
                cellDict[alcls].set_facecolor(color_list[0])
            elif 0.2 < float(cell_value0) <= 0.4:
                cellDict[alcls].set_facecolor(color_list[1])
            elif 0.4 < float(cell_value0) <= 0.6:
                cellDict[alcls].set_facecolor(color_list[2])
            elif 0.6 < float(cell_value0) <= 0.8:
                cellDict[alcls].set_facecolor(color_list[3])
            elif 0.8 < float(cell_value0) <= 1.0:
                cellDict[alcls].set_facecolor(color_list[4])
            else:
                cellDict[alcls].set_facecolor("#FFFFFF")


def remove_value(tablerows, tablecols, mpl_table):
    allcells = [(x, y) for x in tablerows[1:] for y in tablecols[2:]]
    for alcls in allcells:
        mpl_table._cells[alcls]._text.set_text("")


def add_certain_value(tablerows, tablecols, mpl_table, cellDict):
    allcells = [(x, y) for x in tablerows[1:] for y in tablecols[2:]]
    for alcls in allcells:
        # print(cellDict[alcls]._text.get_text())
        cell_value0 = json.loads(cellDict[alcls]._text.get_text())[1]
        mpl_table._cells[alcls]._text.set_text("")
        # cell_value0=(cellDict[alcls]._text.get_text())
        if cell_value0 == -999.0:
            mpl_table._cells[alcls]._text.set_text("")
        elif cell_value0 == 999.0:
            mpl_table._cells[alcls]._text.set_text("")
        else:
            ncl = "%.1f" % cell_value0
            mpl_table._cells[alcls]._text.set_text(ncl)


def aset_height_for_row_except_head(table, rowlist, height):
    cells_list = []
    for row in rowlist:
        cells = [key for key in table._cells if key[0] == row]
        cells_list.append(cells)
    for cells in cells_list:
        for cell in cells:
            table._cells[cell]._height = height


def bset_height_for_row_except_head(table, rowlist, height):
    for row in rowlist:
        for col in range(len(table[row])):
            cell = table[row, col]
            cell._height = height


def cset_height_for_row_except_head(table, row_height):
    """chatGPT function"""
    for i, cell in six.iteritems(table._cells):
        if i[0] == 0:  # Skip header row
            continue
        cell.set_height(row_height)


def set_height_for_row_except_head(cellDict, header_row_count, height):
    for cell_key, cell in cellDict.items():
        row, col = cell_key
        if row < header_row_count:
            continue  # skip header rows
        cell.set_height(height)


def table_header_colour(tablerows, tablecols, cellDict, mpl_table):
    allcells = [(x, y) for x in tablerows[0:1] for y in tablecols]
    header_list = [
        "Region",
        "SPI",
        "Jul",
        "Aug",
        "Sep",
        "",
        "Jul",
        "Aug",
        "Sep",
        "",
        "Jul",
        "Aug",
        "Sep",
        "Oct",
        "",
        "Nov",
        "Dec",
        "Jan",
        "Feb",
        "Mar",
        "Apr",
        "May",
        "Jun",
        "Jul",
        "Aug",
        "Sep",
        "Oct",
        "",
        "Nov",
        "Dec",
        "Jan",
        "Feb",
        "Mar",
        "Apr",
        "May",
        "Jun",
        "Jul",
        "Aug",
        "Sep",
        "Oct",
        "",
    ]
    for idx, alcls in enumerate(allcells):
        cellDict[alcls].set_facecolor("#FFFFFF")
        print(header_list[idx])
        text = header_list[idx]
        mpl_table._cells[alcls]._text.set_text(text)


### funciton for table creation
def render_mpl_table(
    data,
    color_list,
    col_width=1.0,
    row_height=0.425,
    font_size=5,
    header_color="#40466e",
    row_colors=["#f1f1f2", "w"],
    edge_color="w",
    bbox=[0, 0, 1, 1],
    header_columns=0,
    ax=None,
    **kwargs,
):
    """
    Renders a matplotlib table from a pandas DataFrame, allowing for customization of various aesthetic parameters.

    Parameters:
    - data (pandas.DataFrame): The data to display in the table.
    - color_list (list): A list of colors to use for cell background coloring based on cell values.
    - col_width (float): The width of the columns. Default is 1.0.
    - row_height (float): The height of the rows. Default is 0.625.
    - font_size (int): Font size for the cell texts. Default is 5.
    - header_color (str): Color code or name for the table header's background. Default is '#40466e'.
    - row_colors (list): A list containing color codes for alternating row colors. Default is ['#f1f1f2', 'w'].
    - edge_color (str): Color code or name for the cell edge lines. Default is 'w' (white).
    - bbox (list): A 4-element list defining the bounding box of the table within the plot. Default is [0, 0, 1, 1].
    - header_columns (int): The number of initial columns considered as header columns. Default is 0.
    - ax (matplotlib.axes.Axes): The matplotlib axes object where the table will be rendered. If None, a new one will be created.

    Returns:
    - ax (matplotlib.axes.Axes): The matplotlib axes object with the rendered table.

    This function creates a visual representation of a DataFrame as a static table in a matplotlib figure. It allows for
    significant customization, including cell coloring based on values, flexible sizing, font adjustments, and more. The
    function is particularly useful for creating detailed reports or visual summaries of data within a matplotlib figure.
    
    Additional keyword arguments (**kwargs) are passed directly to the `matplotlib.axes.Axes.table` method.
    """
    mpl_table = ax.table(
        cellText=data.values, bbox=bbox, colLabels=[""] * 42, cellLoc="center", **kwargs
    )
    set_align_for_column(mpl_table, col=0, align="left")
    set_width_for_column(mpl_table, 0, 0.6)
    set_width_for_column(mpl_table, 1, 0.5)
    for idx in range(2, 42):
        set_width_for_column(mpl_table, idx, 0.2)
    set_height_for_row(mpl_table, 0, 0.01)
    # set_height_for_row_except_head(mpl_table, np.arange(1, len(data.index)), 0.06)
    # set_height_for_row_except_head(mpl_table, row_height=0.03)
    cellDict = mpl_table.get_celld()
    set_height_for_row_except_head(cellDict, header_row_count=1, height=0.03)
    mpl_table.auto_set_font_size(False)
    mpl_table.set_fontsize(font_size)
    cellDict = mpl_table.get_celld()
    tablerows = np.arange(0, len(data.index) + 1)
    tablecols = np.arange(0, len(data.columns))
    for k, cell in six.iteritems(mpl_table._cells):
        cell.set_edgecolor(edge_color)
        if k[0] == 0 or k[1] < header_columns:
            cell.set_text_props(weight="bold", color="black")
            cell.set_facecolor(header_color)
        else:
            cell.set_facecolor(row_colors[k[0] % len(row_colors)])
    colorcell(tablerows, tablecols, cellDict, color_list)
    headings = data.columns
    plt.text(
        0.245,
        1.08,
        "Moderate",
        fontsize=10,
        fontweight="bold",
        color="black",
        ha="left",
        va="center",
        transform=ax.transAxes,
    )
    plt.text(
        0.545,
        1.08,
        "Severe",
        fontsize=10,
        fontweight="bold",
        color="black",
        ha="left",
        va="center",
        transform=ax.transAxes,
    )
    plt.text(
        0.845,
        1.08,
        "Extreme",
        fontsize=10,
        fontweight="bold",
        color="black",
        ha="left",
        va="center",
        transform=ax.transAxes,
    )
    table_header_colour(tablerows, tablecols, cellDict, mpl_table)
    add_certain_value(tablerows, tablecols, mpl_table, cellDict)
    return ax


def plot_data_table(data_table, stat_var,req_list):
    # Width and height of A4 portrait with 1-inch margins
    width = 3.67 - 2  # one inch margin on each side
    height = 11.69 - 2  # one inch margin on the top and bottom
    fig = plt.figure()
    fig.set_size_inches(height, width)
    # [left, bottom, width, height]
    table = fig.add_axes([0.04, 0.15, 0.93, 0.75], frame_on=False)
    table.xaxis.set_ticks_position("none")
    table.yaxis.set_ticks_position("none")
    table.set_xticklabels("")
    table.set_yticklabels("")
    #######
    laxes = fig.add_axes([0.22, 0.01, 0.4, 0.3], frame_on=False, zorder=0)
    laxes.xaxis.set_ticks_position("none")
    laxes.yaxis.set_ticks_position("none")
    laxes.set_xticklabels("")
    laxes.set_yticklabels("")
    if stat_var == "FAR":
        legend_title = "False Alarm Ratio %"
        color_list = ["#009600", "#64C800", "#ffff00", "#ff7800", "#ff0000"]
    else:
        legend_title = "Hit Rate %"
        color_list = ["#ff0000", "#ff7800", "#ffff00", "#64C800", "#009600"]
    legend_maker(laxes, color_list, legend_title)
    #####
    data1 = data_table[req_list]
    print(data1.info())
    mpl_table = render_mpl_table(
        data1, color_list, header_columns=0, col_width=0.2, ax=table
    )
    # cellDict = mpl_table.get_celld()
    # set_height_for_row_except_head(cellDict, header_row_count=1, height=0.06)
    # set_height_for_row_except_head(mpl_table, row_height=0.125)
    var = stat_var.lower()
    plt.savefig(f"{latex_path}{var}_{region_id}_prob_v20240515.jpg", dpi=300)
    
def pass_month_get_colnames(months):
    original_list = [
            "region_x",
            "season",
            "nov_x",
            "dec_x",
            "jan_x",
            "feb_x",
            "mar_x",
            "apr_x",
            "may_x",
            "jun_x",
            "jul_x",
            "aug_x",
            "sep_x",
            "oct_x",
            "empty1",
            "nov_y",
            "dec_y",
            "jan_y",
            "feb_y",
            "mar_y",
            "apr_y",
            "may_y",
            "jun_y",
            "jul_y",
            "aug_y",
            "sep_y",
            "oct_y",
            "empty2",
            "nov_z",
            "dec_z",
            "jan_z",
            "feb_z",
            "mar_z",
            "apr_z",
            "may_z",
            "jun_z",
            "jul_z",
            "aug_z",
            "sep_z",
            "oct_z",
        ]
    #months = ['jul', 'aug', 'sep']
    suffixes = ['_x', '_y', '_z']

    organized_list = ['region_x', 'season',]

    for suffix in suffixes:
        for month in months:
            item = month + suffix
            if item in original_list:
                organized_list.append(item)

        if suffix == '_x':
            organized_list.append('empty1')
        elif suffix == '_y':
            organized_list.append('empty2')
    return organized_list


def table_df(tab_df,stat_var):
    tab_df_a=tab_df.rename(columns={"lead_time":"lt"})
    tab_df_m=create_month_column(tab_df_a)
    tab_df_m['pod_v'] = tab_df_m.apply(lambda x: [x['hit_rates'], x['trigger_values']], axis=1)
    tab_df_m['far_v'] = tab_df_m.apply(lambda x: [x['false_alarm_ratios'], x['trigger_values']], axis=1)
    tab_df_m['pod_v'] = tab_df_m['pod_v'].apply(lambda x: round_list(x, 2))
    tab_df_m['far_v'] = tab_df_m['far_v'].apply(lambda x: round_list(x, 2))
    mapping_dict = {0: 'Karamoja', 1: 'Marsabit', 2: 'Wajir'}
    tab_df_m['region_x'] = tab_df_m['region'].replace(mapping_dict)
    p = tab_df_m.pivot_table(index=['region_x', 'season'], 
                            columns='new_column', 
                            values='pod_v', 
                            aggfunc='first')
    pf = p.reset_index()
    # Apply the custom function to each cell in the DataFrame
    pf1 = pf.applymap(replace_with_list)
    pf1.columns.name = None
    pf1['empty1']=[[-999.0, -999.0]]* len(pf1)
    pf1['empty2']=[[-999.0, -999.0]]* len(pf1)
    months = ['jul', 'aug', 'sep']
    organized_list=pass_month_get_colnames(months)
    pf2=pf1[organized_list]
    mask = (pf2['region_x'].isin(['Marsabit', 'Wajir'])) & (pf2['season'] == 'OND')
    pf3 = pf2[mask]
    plot_data_table(pf3,stat_var,organized_list)
    


# Region 1, Marsabit

In [ ]:
region_id=1
season_str='OND'
sc_season_str=season_str.lower()
row_annotations=make_barchart_annotations()
threshold_dict=get_threshold(region_id, sc_season_str)


lead_int=2
obs_df,fct_df,metrix_df_lt2,dec_dict,dec_df2,plot_df=get_mean_ens_triggers(region_id,season_str,lead_int)
obs_plot=obs_chart_with_triggers('obs',obs_df, 'year', 'spi3', threshold_dict,row_annotations[2])
lt2_plot=obs_chart_with_triggers('fct',plot_df, 'year', 'ep_pb', dec_dict,row_annotations[1])


lead_int=3
obs_df,fct_df,metrix_df_lt3,dec_dict,dec_df3,plot_df=get_mean_ens_triggers(region_id,season_str,lead_int)
lt3_plot=obs_chart_with_triggers('fct',plot_df, 'year', 'ep_pb', dec_dict,row_annotations[2])

lead_int=4
obs_df,fct_df,metrix_df_lt4,dec_dict,dec_df4,plot_df=get_mean_ens_triggers(region_id,season_str,lead_int)
lt4_plot=obs_chart_with_triggers('fct',plot_df, 'year', 'ep_pb', dec_dict,row_annotations[3])

dec_lt2=dec_df2.drop_duplicates('cat')
dec_lt3=dec_df3.drop_duplicates('cat')
dec_lt4=dec_df4.drop_duplicates('cat')
tab_df=pd.concat([dec_lt2,dec_lt3,dec_lt4])
tab_df1=tab_df.reset_index().rename(columns={'index': 'idx'})
tab_df2=tab_df1[['lead_time','trigger_values','cat','trigger_values','hits','misses','FA','CN']]
tab_df2=tab_df2.rename(columns={"trigger_values": "Trigger"})
tab_df2=tab_df2.round({'Trigger': 1})
tab_df2['Trigger']=tab_df2['Trigger'].astype(str)

#tab_df3=tab_df2[['lead_time','Trigger','cat']]

tab_plot=table(tab_df2).properties(height=200,width=400)
#tab_plot


emtpy_plot=alt.Chart(pd.DataFrame({'A': []})).mark_text().encode().properties(
    width=400,
    height=200)

panels = alt.vconcat(
    alt.hconcat(obs_plot,lt2_plot),
    alt.hconcat(tab_plot, lt3_plot),
    alt.hconcat(emtpy_plot, lt4_plot),
)

panels.configure_view(stroke=None).configure_axisY(
    labelFontSize=12,
    titleFontSize=14
).configure_axisX(
    labelFontSize=10,
    titleFontSize=12
).configure_legend(
    labelFontSize=12,
    titleFontSize=14
)

panels.save(f"{latex_path}{region_id}-{sc_season_str}_v20240515.png")
#panels

In [ ]:
tab_df_a=tab_df.rename(columns={"lead_time":"lt"})
tab_df_m=create_month_column(tab_df_a)


tab_df_m['pod_v'] = tab_df_m.apply(lambda x: [x['hit_rates'], x['trigger_values']], axis=1)
tab_df_m['far_v'] = tab_df_m.apply(lambda x: [x['false_alarm_ratios'], x['trigger_values']], axis=1)

tab_df_m['pod_v'] = tab_df_m['pod_v'].apply(lambda x: round_list(x, 2))
tab_df_m['far_v'] = tab_df_m['far_v'].apply(lambda x: round_list(x, 2))


mapping_dict = {0: 'Karamoja', 1: 'Marsabit', 2: 'Wajir'}

tab_df_m['region_x'] = tab_df_m['region'].replace(mapping_dict)

p = tab_df_m.pivot_table(index=['region_x', 'season'], 
                            columns='new_column', 
                            values='pod_v', 
                            aggfunc='first')

pf = p.reset_index()


# Apply the custom function to each cell in the DataFrame
pf1 = pf.applymap(replace_with_list)

pf1.columns.name = None

pf1['empty1']=[[-999.0, -999.0]]* len(pf1)
pf1['empty2']=[[-999.0, -999.0]]* len(pf1)

months = ['jul', 'aug', 'sep']

organized_list=pass_month_get_colnames(months)

pf2=pf1[organized_list]
# pivoted_df1.to_csv(f'{data_path}pod.csv')
stat_var='POD'
mask = (pf2['region_x'].isin(['Marsabit', 'Wajir'])) & (pf2['season'] == 'OND')
pf3 = pf2[mask]
plot_data_table(pf3,stat_var,organized_list)


stat_var='FAR'
mask = (pf2['region_x'].isin(['Marsabit', 'Wajir'])) & (pf2['season'] == 'OND')
pf3 = pf2[mask]
plot_data_table(pf3,stat_var,organized_list)

In [ ]:
pf3

In [ ]:
df0=metrix_df_lt2
df=df0[df0['auroc_scores']>=0.5]
df1=df.round(1)
df1.columns=['region', 'season', 'cat', 'year', 'tri. val', 'thres.',
       'dry-seas', 'hits', 'misses', 'FA', 'CN', 'POD',
       'FAR', 'Bias S.', 'HKS',
       'HSS', 'auroc s.', 'a. lb', 'a. ub']

df1.drop(columns=['region','season','a. lb'], inplace=True)

cat_order = ['ext', 'sev', 'mod']

# Convert 'cat' column to categorical with the desired order
df1['cat'] = pd.Categorical(df1['cat'], categories=cat_order, ordered=True)
#df_sorted = df1.sort_values(by='cat')
df_sorted1=df1.sort_values(by=['cat','tri. val'])
#df_sorted2=df_sorted1.sort_values(by='cat')

region_name=mapping_dict[region_id] 
latex_table = df_sorted1.to_latex(index=False, longtable=True,float_format='%.2f',caption=f'Verification metrices for region {region_name} for lead time 2, Sep')

latex_document = f"""
\\documentclass{{article}}
\\usepackage{{longtable}}
\\usepackage{{booktabs}}
\\usepackage{{geometry}}
\\geometry{{a4paper,
    left=1cm,
    right=1cm,
    top=1cm,
    bottom=1cm,
}}
\\begin{{document}}
{latex_table}
\\end{{document}}
"""

with open(f"{latex_path}vm_{region_id}_lt2.tex", "w") as f:
    f.write(latex_document)

In [ ]:
df0=metrix_df_lt3
df=df0[df0['auroc_scores']>=0.5]
df1=df.round(1)
df1.columns=['region', 'season', 'cat', 'year', 'tri. val', 'thres.',
       'dry-seas', 'hits', 'misses', 'FA', 'CN', 'POD',
       'FAR', 'Bias S.', 'HKS',
       'HSS', 'auroc s.', 'a. lb', 'a. ub']

df1.drop(columns=['region','season','a. lb'], inplace=True)

cat_order = ['ext', 'sev', 'mod']

# Convert 'cat' column to categorical with the desired order
df1['cat'] = pd.Categorical(df1['cat'], categories=cat_order, ordered=True)
#df_sorted = df1.sort_values(by='cat')
df_sorted1=df1.sort_values(by=['cat','tri. val'])
#df_sorted2=df_sorted1.sort_values(by='cat')

region_name=mapping_dict[region_id] 
latex_table = df_sorted1.to_latex(index=False, longtable=True,float_format='%.2f',caption=f'Verification metrices for region {region_name} for lead time 3, Aug')

latex_document = f"""
\\documentclass{{article}}
\\usepackage{{longtable}}
\\usepackage{{booktabs}}
\\usepackage{{geometry}}
\\geometry{{a4paper,
    left=1cm,
    right=1cm,
    top=1cm,
    bottom=1cm,
}}
\\begin{{document}}
{latex_table}
\\end{{document}}
"""

with open(f"{latex_path}vm_{region_id}_lt3.tex", "w") as f:
    f.write(latex_document)

In [ ]:
df0=metrix_df_lt4
df=df0[df0['auroc_scores']>=0.5]
df1=df.round(1)
df1.columns=['region', 'season', 'cat', 'year', 'tri. val', 'thres.',
       'dry-seas', 'hits', 'misses', 'FA', 'CN', 'POD',
       'FAR', 'Bias S.', 'HKS',
       'HSS', 'auroc s.', 'a. lb', 'a. ub']

df1.drop(columns=['region','season','a. lb'], inplace=True)

cat_order = ['ext', 'sev', 'mod']

# Convert 'cat' column to categorical with the desired order
df1['cat'] = pd.Categorical(df1['cat'], categories=cat_order, ordered=True)
#df_sorted = df1.sort_values(by='cat')
df_sorted1=df1.sort_values(by=['cat','tri. val'])
#df_sorted2=df_sorted1.sort_values(by='cat')

region_name=mapping_dict[region_id] 
latex_table = df_sorted1.to_latex(index=False, longtable=True,float_format='%.2f',caption=f'Verification metrices for region {region_name} for lead time 4, Jul')

latex_document = f"""
\\documentclass{{article}}
\\usepackage{{longtable}}
\\usepackage{{booktabs}}
\\usepackage{{geometry}}
\\geometry{{a4paper,
    left=1cm,
    right=1cm,
    top=1cm,
    bottom=1cm,
}}
\\begin{{document}}
{latex_table}
\\end{{document}}
"""

with open(f"{latex_path}vm_{region_id}_lt4.tex", "w") as f:
    f.write(latex_document)

In [ ]:
import subprocess
import os

os.chdir(latex_path)
subprocess.run(["pdflatex", f"{latex_path}vm_{region_id}_lt2.tex"])
subprocess.run(["pdflatex", f"{latex_path}vm_{region_id}_lt3.tex"])
subprocess.run(["pdflatex", f"{latex_path}vm_{region_id}_lt4.tex"])

In [ ]:
\documentclass{article}
\usepackage{graphicx}
\usepackage{pdfpages}
\usepackage{geometry}
\geometry{a4paper,
	left=1cm,
	right=1cm,
	top=1cm,
	bottom=1cm,
}
\begin{document}
%	\begin{titlepage}
%		\centering
%		{\Huge MyTitle} % Replace "MyTitle" with your desired word
%		\vfill
%
%		\vfill
%	\end{titlepage}
	
	
	
	\clearpage
	
	\section{PNG Image with Caption}
	
\begin{figure}[htbp]
		\centering
		\includegraphics[width=1\textwidth]{pod_1_prob_v20240515.jpg} % Replace "example.png" with the filename of your PNG image
		\caption{Marsabit selected trigger for OND}
	\end{figure}

    \begin{figure}[htbp]
    	\centering
    	\includegraphics[width=1\textwidth]{far_1_prob_v20240515.jpg} % Replace "example.png" with the filename of your PNG image
    	\caption{Marsabit selected trigger for OND}
    \end{figure}

	\section{Bar chart of selected triggers}
	
	% Use the attachfile package to attach the PDF file

	% Add the PDF file as an attachment
	\begin{figure}[htbp]
	\centering
	\includegraphics[width=1\textwidth]{1-ond_v20240515.png} % Replace "example.png" with the filename of your PNG image
	\caption{Marsabit selected trigger for OND}
    \end{figure}

\newpage
\includepdf[pages=-]{vm_1_lt2.pdf}
\includepdf[pages=-]{vm_1_lt3.pdf}
\includepdf[pages=-]{vm_1_lt4.pdf}

	
\end{document}